In [2]:
import subprocess
import numpy as np
import sys

In [3]:
# PHYSICAL PARAMETERS

# GEOMETRY (all lengths in nm)
r_pore = 30 
r_res = 300
L_pore = 10
L_res = 400

# SOLUTION PROPERTIES (used for setting mesh size near charged surfaces)
sigma = 0.01    # Surface charge in C/m^2
c_bulk = 0.5   # Bulk electrolyte concentration in mol/m^3
epsilon_r = 80  # Relative permitivity
D = 2.0E-9      # Diffusivity of both ion species.

# PHYSICAL CONSTANTS
kBT = 4.1E-21           # Thermal energy at room temp in J
e_charge = 1.6E-19      # elementary charge in C
epsilon_0 = 8.85E-12    # permitivity of free space in C/VM
N_A = 6.22E23           # Avagadro's Number

# DERIVED QUANTITY
l_Deb = np.sqrt(epsilon_r * epsilon_0 * kBT / (2 * N_A * c_bulk * e_charge**2))

In [4]:
# Define the normalized unit system
r0 = r_pore*1.0E-9       # unit length, equal to nanopore radius
c0 = c_bulk              # unit ion concentration, set by bulk concentration
V0 = kBT/e_charge        # unit voltage, set by the thermal voltage
D0 = D                   # unit diffusivity, set by ion diffusivity
epsilon0 = epsilon_0     # unit permitivity, set by vacuum permitivity

t0 = r0**2 / D0          # unit time, set by characteristic diffusion time over unit length
e0 = epsilon0 * V0 * r0  # unit charge




# Convert physical parameters to normalized values
r_pore_norm = r_pore *1.0E-9 / r0
r_res_norm = r_res * 1.0E-9 / r0
L_pore_norm = L_pore * 1.0E-9 / r0
L_res_norm = L_res * 1.0E-9 / r0

sigma_norm = sigma * (r0**2 / e0)
c_bulk_norm = c_bulk / c0
D_norm = D/D0

l_Deb_norm = l_Deb / r0


# Print out the normalized values. Ideally, they will all be within a few orders of unity
print(f"normalized pore radius: {r_pore_norm}")
print(f"normalized reservoir radius: {r_res_norm}")
print(f"normalized pore length: {L_pore_norm}")
print(f"normalized reservoir length: {L_res_norm}")

print(f"normalized surface charge: {sigma_norm}")
print(f"normalized bulk concentration: {c_bulk_norm}")
print(f"normalized diffusivity: {D_norm}")

print(f"normalized debeye length: {l_Deb_norm}")



normalized pore radius: 1.0
normalized reservoir radius: 10.0
normalized pore length: 0.3333333333333333
normalized reservoir length: 13.333333333333332
normalized surface charge: 1322.860686233981
normalized bulk concentration: 1.0
normalized diffusivity: 1.0
normalized debeye length: 0.45006177377595874


In [ ]:
# Write all the relevant parameter values into a file
with open("dumbell_params.dat", "w") as f:

    f.write("# This file contains information about the physical parameters for a simulated nanopore")

    f.write("# GEOMETRIC PARAMETERS (in nm)\n")
    f.write("pore_radius " + str(r_pore) + "\n")
    f.write("pore_length " + str(L_pore) + "\n")
    f.write("reservoir_radius " + str(r_res) + "\n")
    f.write("reservoir_length " + str(L_pore) + "\n")

    f.write("\n# CHEMICAL PARAMETERS (in SI units)\n")
    f.write("surface_charge " + str(sigma) + "\n")
    f.write("bulk_concentration " + str(c_bulk) + "\n")
    f.write("relative_permitivity " + str(epsilon_r) + "\n")
    f.write("diffusivity " + str(D) + "\n")

    f.write("\n# NORMALIZATION (in SI units)\n")
    f.write("unit_length " + str(r0) + "\n")
    f.write("unit_concentration " + str(c0) + "\n")
    f.write("unit_voltage " + str(V0) + "\n")
    f.write("unit_diffusivity " + str(D0) + "\n")
    f.write("unit_permitivity " + str(epsilon0) + "\n")
    f.write("unit_time " + str(t0) + "\n")
    f.write("unit_charge " + str(e0) + "\n")


In [6]:
# Use Gmsh to run the script dumbell.geo with the normalized values we computed
subprocess.run([
    "gmsh", 
    "dumbell.geo",                          # name of the .geo script to run
    "-3",                                   # Generate a 3D mesh
    "-setnumber", "lpn", str(L_pore_norm),  
    "-setnumber", "rrn", str(r_res_norm),
    "-setnumber", "lrn", str(L_res_norm),
    "-setnumber", "ldn", str(l_Deb_norm),
    "-format", "msh2",                      # FiPy is only compatible with the older mesh format
    "-o", "dumbell.msh",                    # Name of the mesh file we create
], check=True)

CompletedProcess(args=['gmsh', 'dumbell.geo', '-3', '-setnumber', 'lpn', '0.3333333333333333', '-setnumber', 'rrn', '10.0', '-setnumber', 'lrn', '13.333333333333332', '-setnumber', 'ldn', '0.45006177377595874', '-format', 'msh2', '-o', 'dumbell.msh'], returncode=0)